In [ ]:
!pip install transformers torchaudio


In [11]:
import torch
import torchaudio
from transformers import Wav2Vec2Model, Wav2Vec2Processor
import tenseal as ts
import utils  

audio_file1 = "my_data/java_x3.wav"
audio_file2 = "my_data/python_java_cetic_1s_mathieu_voice.wav"

# Chargement du modèle Wav2Vec2 préentraîné (base multilingue) 
model_name = "facebook/wav2vec2-base-960h"
processor = Wav2Vec2Processor.from_pretrained(model_name)
model = Wav2Vec2Model.from_pretrained(model_name)
model.eval()

# Fonction pour encoder un fichier audio en embedding ===
def extract_embedding_wav2vec2(file_path):
    waveform, sample_rate = torchaudio.load(file_path)
    if sample_rate != 16000:
        waveform = torchaudio.functional.resample(waveform, orig_freq=sample_rate, new_freq=16000)
    inputs = processor(waveform.squeeze(), sampling_rate=16000, return_tensors="pt")
    with torch.no_grad():
        outputs = model(**inputs)
    embedding = outputs.last_hidden_state.mean(dim=1).squeeze().tolist()  # Moyenne temporelle
    return embedding

print("audio_file1 :", audio_file1)
print("audio_file2 :", audio_file2)
vec1 = extract_embedding_wav2vec2(audio_file1)
print("Embedding pour l'audio 1 (Céline) :", vec1)
print("Taille de l'embedding pour l'audio 1 :", len(vec1))
vec2 = extract_embedding_wav2vec2(audio_file2)
print("Embedding pour l'audio 2 (Mathieu) :", vec2)
print("Taille de l'embedding pour l'audio 2 :", len(vec2))
context = ts.context(
    ts.SCHEME_TYPE.CKKS,
    poly_modulus_degree=8192,
    coeff_mod_bit_sizes=[60, 40, 40, 60]
)
context.global_scale = 2**40
context.generate_galois_keys()

enc_vec1 = ts.ckks_vector(context, vec1)
enc_vec2 = ts.ckks_vector(context, vec2)

my_secret_key = context.serialize(save_secret_key=True)
utils.write_data("Keys_provider/secret_key.txt", my_secret_key)
context.make_context_public()
public_key = context.serialize()
utils.write_data("Keys_provider/public_key.txt", public_key)




utils.write_data("output_provider/python_java_cetic_1s_celine_encrypted.txt", enc_vec1.serialize())
utils.write_data("output_provider/python_java_cetic_1s_mathieu_encrypted.txt", enc_vec2.serialize())





Some weights of Wav2Vec2Model were not initialized from the model checkpoint at facebook/wav2vec2-base-960h and are newly initialized: ['masked_spec_embed']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


audio_file1 : my_data/java_x3.wav
audio_file2 : my_data/python_java_cetic_1s_mathieu_voice.wav
Embedding pour l'audio 1 (Céline) : [-0.019798332825303078, 0.016815342009067535, 0.11744557321071625, -0.027278613299131393, 0.12836749851703644, -0.0600619912147522, 0.12534436583518982, -0.0422908216714859, 0.12339404225349426, -0.2980995774269104, -0.04810024052858353, 0.09929981082677841, 0.004297266714274883, 0.010302015580236912, -0.14530149102210999, -0.04320979490876198, -0.22524334490299225, 0.260630339384079, 0.01467843260616064, 0.07179002463817596, -0.15579165518283844, 0.04793282970786095, 0.1294289231300354, -0.00036087867920286953, 0.10675051063299179, -0.06669510155916214, -0.16987107694149017, -0.022042226046323776, -0.09346601366996765, -0.10126955807209015, 0.05397844314575195, -0.01183465588837862, -0.12035977095365524, -0.09985210746526718, -0.23408764600753784, -0.18993209302425385, -0.022587040439248085, -0.26745864748954773, -0.1153259128332138, -0.05965963378548622, 

In [12]:
secret_key_data = utils.read_data("Keys_provider/secret_key.txt")
context = ts.context_from(secret_key_data)
import math
# Vérification de la clé privée
if not context.is_private():
    raise ValueError("Clé secrète manquante pour le déchiffrement.")

# === 1. Distance euclidienne ===
diff_data = utils.read_data("outputs_manipulator/difference.txt")
diff_vec = ts.ckks_vector_from(context, diff_data)
diff_decrypted = diff_vec.decrypt()
distance_squared = sum(diff_decrypted)
distance = math.sqrt(distance_squared)

# === 2. Produit scalaire (dot product) ===
dot_data = utils.read_data("outputs_manipulator/dot_product.txt")
dot_vec = ts.ckks_vector_from(context, dot_data)
dot_decrypted = dot_vec.decrypt()
dot_product = sum(dot_decrypted)

# === 3. Normes (||vec1||² et ||vec2||²) ===
norm1_data = utils.read_data("outputs_manipulator/norm1_squared.txt")
norm2_data = utils.read_data("outputs_manipulator/norm2_squared.txt")

norm1_vec = ts.ckks_vector_from(context, norm1_data)
norm2_vec = ts.ckks_vector_from(context, norm2_data)

norm1 = math.sqrt(sum(norm1_vec.decrypt()))
norm2 = math.sqrt(sum(norm2_vec.decrypt()))

# === 4. Corrélation cosinus ===
if norm1 != 0 and norm2 != 0:
    correlation = dot_product / (norm1 * norm2)
else:
    correlation = 0.0  # Pour éviter la division par zéro

# === 5. Affichage des résultats ===
print("\n Résultats déchiffrés :")
print(f"Distance euclidienne : {distance:.4f}")
print(f"Corrélation cosinus  : {correlation:.4f}")

# === 6. Interprétation ===
print("\n Interprétation :")
print("Distance euclidienne :", distance)
if distance < 2: 
    print(" Les audios sont très similaires (distance faible)")
else:
    print(" Les audios semblent différents (distance élevée)")

print("Corrélation cosinus :", correlation)
if correlation > 0.9:
    print(" Forte corrélation : les audios sont très similaires")
elif correlation > 0.5:
    print(" Corrélation modérée : similarité partielle")
else:
    print(" Faible corrélation : audios probablement différents")


 Résultats déchiffrés :
Distance euclidienne : 4.1420
Corrélation cosinus  : 0.7943

 Interprétation :
Distance euclidienne : 4.142010538834153
 Les audios semblent différents (distance élevée)
Corrélation cosinus : 0.7943305783076026
 Corrélation modérée : similarité partielle
